In [ ]:
# ============================================
# CROSS-DATASET EVALUATION: DFDC02 → Celeb-DF v2
# ============================================
# Required Kaggle inputs:
#   - Pretrained checkpoint (DFDC02-only, T=16 OR T=32)
#   - celebdf-preprocessed (T matching the checkpoint)
#
# This notebook loads a model trained on DFDC02 only,
# and evaluates it on Celeb-DF v2 test set (full cross-domain transfer).
#
# Expected outcome (from DeepfakeBench): AUC drop 15-25% vs in-domain.

import subprocess, sys, os

# Step 1: Install dependencies
print('=== Installing dependencies ===')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                '--force-reinstall', 'Pillow==10.4.0'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'numpy<2', 'scipy<1.17', 'scikit-learn', 'timm', 'facenet-pytorch'],
               check=True)
print('Dependencies installed.')

# Step 2: Clone project
if not os.path.exists('/kaggle/working/project/.git'):
    subprocess.run(['rm', '-rf', '/kaggle/working/project'], check=False)
    subprocess.run(['git', 'clone', 'https://github.com/Jokeera/deepfake-detection.git',
                     '/kaggle/working/project'], check=True)
else:
    subprocess.run(['git', '-C', '/kaggle/working/project', 'pull', '--ff-only'], check=True)

# Step 3: Write evaluation script
eval_script = r'''
import os, sys, json, glob, time
os.chdir('/kaggle/working/project')
sys.path.insert(0, '/kaggle/working/project')

import torch
import numpy as np
from torch.utils.data import DataLoader

from config import Config
from dataset import VideoDataset, build_video_index
from models.dual_path import DualPathModel
from models.spatial_only import SpatialOnlyModel
from models.temporal_only import TemporalOnlyModel
from models.sequential import SequentialModel
from utils import calculate_metrics

# ============ CONFIG ============
# Path to your DFDC02-trained checkpoint
CHECKPOINT_PATH = None
for p in glob.glob('/kaggle/input/**/best_model.pt', recursive=True):
    if 'dfdc02' in p.lower() and 'full' in p.lower() and 'celeb' not in p.lower() and 'dfd01' not in p.lower():
        CHECKPOINT_PATH = p
        break

assert CHECKPOINT_PATH is not None, 'DFDC02-only checkpoint not found in /kaggle/input/. Upload it as a dataset.'
print(f'Using checkpoint: {CHECKPOINT_PATH}')

# Find Celeb-DF preprocessed
CELEBDF_PATH = None
for root, dirs, files in os.walk('/kaggle/input'):
    if 'celeb' in root.lower() and 'real' in dirs and 'fake' in dirs:
        CELEBDF_PATH = root
        break

assert CELEBDF_PATH is not None, 'celebdf preprocessed not found in /kaggle/input/'
print(f'Celeb-DF path: {CELEBDF_PATH}')

# ============ LOAD CHECKPOINT ============
ckpt = torch.load(CHECKPOINT_PATH, map_location='cuda', weights_only=False)
saved_cfg = ckpt.get('config', {})

cfg = Config()
cfg.dataset_root = CELEBDF_PATH
cfg.dataset_name = 'celebdf'
cfg.model_type = saved_cfg.get('model_type', 'full')
cfg.fusion_type = saved_cfg.get('fusion_type', 'adaptive')
cfg.num_frames = saved_cfg.get('num_frames', 16)
cfg.batch_size = 8
cfg.device = 'cuda'

print(f'Model: model_type={cfg.model_type}, fusion={cfg.fusion_type}, T={cfg.num_frames}')

# ============ BUILD MODEL ============
if cfg.model_type == 'full':
    model = DualPathModel(cfg)
elif cfg.model_type == 'spatial':
    model = SpatialOnlyModel(cfg)
elif cfg.model_type == 'temporal':
    model = TemporalOnlyModel(cfg)
elif cfg.model_type == 'sequential':
    model = SequentialModel(cfg)
else:
    raise ValueError(f'Unknown model_type: {cfg.model_type}')

model.load_state_dict(ckpt['model_state_dict'])
model = model.cuda().eval()
print(f'Loaded model: {sum(p.numel() for p in model.parameters())/1e6:.2f}M params')

# ============ LOAD CELEB-DF DATA ============
video_index = build_video_index(cfg.dataset_root, cfg=cfg)
print(f'Total Celeb-DF videos: {len(video_index)}')

# Use all videos as test (no split — we want full cross-domain transfer measurement)
ds = VideoDataset(video_index, cfg, is_training=False)
loader = DataLoader(ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)

# ============ INFER ============
all_preds, all_labels = [], []
t0 = time.time()
with torch.no_grad():
    for i, batch in enumerate(loader):
        frames = batch['frames'].cuda(non_blocking=True)
        diffs = batch['diff_frames'].cuda(non_blocking=True) if 'diff_frames' in batch else None
        labels = batch['label']

        if cfg.model_type == 'full':
            logits, alpha = model(frames, diffs)
        elif cfg.model_type == 'spatial':
            logits = model(frames)
        elif cfg.model_type == 'temporal':
            logits = model(diffs)
        elif cfg.model_type == 'sequential':
            logits = model(frames)

        probs = torch.sigmoid(logits).cpu().numpy().flatten()
        all_preds.extend(probs.tolist())
        all_labels.extend(labels.numpy().flatten().tolist())

        if i % 10 == 0:
            print(f'  batch {i}/{len(loader)}, elapsed {time.time()-t0:.1f}s')

# ============ METRICS ============
all_preds = np.array(all_preds)
all_labels = np.array(all_labels)
metrics = calculate_metrics(all_labels, all_preds)

print()
print('=' * 60)
print('CROSS-DATASET EVALUATION: DFDC02 → Celeb-DF v2')
print('=' * 60)
print(f'Videos: {len(all_labels)} (real: {(all_labels==0).sum()}, fake: {(all_labels==1).sum()})')
print(f'AUC:      {metrics["auc"]:.4f}')
print(f'Accuracy: {metrics["accuracy"]:.4f}')
print(f'F1:       {metrics["f1"]:.4f}')
print(f'EER:      {metrics["eer"]:.4f}')

# Save predictions
import csv
with open('/kaggle/working/cross_eval_dfdc02_to_celebdf.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['video_id', 'label', 'prob_fake'])
    for i, (lbl, pred) in enumerate(zip(all_labels, all_preds)):
        w.writerow([f'celebdf_{i:05d}', int(lbl), float(pred)])

with open('/kaggle/working/cross_eval_dfdc02_to_celebdf_metrics.json', 'w') as f:
    json.dump({
        'source_dataset': 'dfdc02',
        'target_dataset': 'celebdf',
        'checkpoint': CHECKPOINT_PATH,
        'metrics': metrics,
        'n_videos': len(all_labels),
        'n_real': int((all_labels==0).sum()),
        'n_fake': int((all_labels==1).sum()),
    }, f, indent=2)

print('Saved: cross_eval_dfdc02_to_celebdf.csv and metrics.json')
'''

with open('/kaggle/working/run_cross_eval.py', 'w') as f:
    f.write(eval_script)

# Step 4: Run
result = subprocess.run(
    [sys.executable, '/kaggle/working/run_cross_eval.py'],
    cwd='/kaggle/working/project',
    env={**os.environ, 'PYTHONPATH': '/kaggle/working/project'},
)
print(f'Subprocess exited: {result.returncode}')
